# **Epoch Selection — BLEU Evaluation**<br>

In [1]:
import pickle
import pandas as pd
from statistics import mean
from modules.BPE_tokenizer import tokenize_eng, tokenize_pol, make_ref_encoder
from modules import Data, Tests

## **v1_2.** translator_v1_2

In [132]:
with open('../local_data/gender_pronouns/1st_person/data_final_1st_person.pkl', 'rb') as f:
    df_data = pickle.load(f)

df_samples = {}
for val in ['F', 'M', 'NA', 'NA_OTHER']:
    df_samples[f'self_{val}'] = df_data[df_data['self_ref'] == val].sample(100, random_state=42).reset_index(drop=True)

for df_sample in df_samples.values():
    df_sample['eng_split'] = df_sample['eng_text'].apply(tokenize_eng)
    df_sample['pol_split'] = df_sample['pol_text'].apply(tokenize_pol)

In [133]:
with open('../local_data/model_reference/translator_v1_2/tokenizer_eng.pkl', 'rb') as f:
    tokenizer_eng = pickle.load(f)
with open('../local_data/model_reference/translator_v1_2/tokenizer_pol.pkl', 'rb') as f:
    tokenizer_pol = pickle.load(f)
with open('../local_data/model_reference/translator_v1_2/encoder_eng.pkl', 'rb') as f:
    encoder_eng = pickle.load(f)
with open('../local_data/model_reference/translator_v1_2/encoder_pol.pkl', 'rb') as f:
    encoder_pol = pickle.load(f)

encode_pol_row = make_ref_encoder(tokenizer_pol, encoder_pol, [('self_ref', {'F': '<self_f>', 'M': '<self_m>', 'NA': '<self_na>', 'NA_OTHER': '<self_na>'})])

for df_sample in df_samples.values():
    df_sample['eng_ids'] = df_sample['eng_split'].apply(lambda tab: encoder_eng.encode_snt(tab) + [tokenizer_eng.vocab['<eos>']])
    df_sample['pol_ids'] = df_sample.apply(encode_pol_row, axis=1)

df_samples = {key: Data.trim_by_ids(df_sample, ['eng_ids', 'pol_ids'], max_len=35) for key, df_sample in df_samples.items()}

Dropped 0 / 100 rows exceeding max_len=35
Dropped 0 / 100 rows exceeding max_len=35
Dropped 0 / 100 rows exceeding max_len=35
Dropped 0 / 100 rows exceeding max_len=35


## **v1_2.** BLEU Per Epoch **(100 x 4)**

In [ ]:
epoch_rows = []
print(f"{'||':>12}   self_F   |   self_M   |   self_NA  | self_OTHER ||  avg_bleu  ||")
for epoch in range(7, 16):
    evaluator = Tests.BleuEvaluation(checkpoint_path=f'../checkpoints/translator_v1_2-{epoch}.pt',
                                      artifacts_path='../local_data/model_reference/translator_v1_2/',
                                      hyperparams=(512, 1024, 8, 4, 0.3, 37), n_ref=1)

    ref_means = {key: mean(evaluator.score_series(df_sample, num_k=4)) for key, df_sample in df_samples.items()}
    row = {'epoch': epoch, 'avg_bleu': mean(ref_means.values()), **ref_means}
    epoch_rows.append(row)
    print(f"Epoch: {row['epoch']:>2} ||  {row['self_F']:.6f}  |  {row['self_M']:.6f}  |  {row['self_NA']:.6f}  |  {row['self_NA_OTHER']:.6f}  ||  {row['avg_bleu']:.6f}  ||")

          ||   self_F   |   self_M   |   self_NA  | self_OTHER ||  avg_bleu  ||


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Epoch: 15 ||  0.439527  |  0.357304  |  0.419295  |  0.339455  ||  0.388895  ||


## **v1_2.** Best epoch according to **avg_bleu**

In [146]:
df_epochs = pd.DataFrame(epoch_rows).sort_values('avg_bleu', ascending=False).reset_index(drop=True)
df_epochs.head(5)

,epoch,avg_bleu,self_F,self_M,self_NA,self_NA_OTHER
0,15,0.388895,0.439527,0.357304,0.419295,0.339455
1,14,0.382488,0.429323,0.340296,0.406937,0.353394
2,13,0.381033,0.420091,0.353802,0.414084,0.336157
3,12,0.365273,0.411039,0.315926,0.403010,0.331117
4,11,0.358831,0.392268,0.330323,0.379948,0.332785


## **v1_3.** translator_v1_3 **- same samples as v1_2**

In [145]:
with open('../local_data/model_reference/translator_v1_3/tokenizer_eng.pkl', 'rb') as f:
    tokenizer_eng = pickle.load(f)
with open('../local_data/model_reference/translator_v1_3/tokenizer_pol.pkl', 'rb') as f:
    tokenizer_pol = pickle.load(f)
with open('../local_data/model_reference/translator_v1_3/encoder_eng.pkl', 'rb') as f:
    encoder_eng = pickle.load(f)
with open('../local_data/model_reference/translator_v1_3/encoder_pol.pkl', 'rb') as f:
    encoder_pol = pickle.load(f)

encode_pol_row = make_ref_encoder(tokenizer_pol, encoder_pol, [('self_ref', {'F': '<self_f>', 'M': '<self_m>', 'NA': '<self_na>', 'NA_OTHER': '<self_na>'})])

for df_sample in df_samples.values():
    df_sample['eng_ids'] = df_sample['eng_split'].apply(lambda tab: encoder_eng.encode_snt(tab) + [tokenizer_eng.vocab['<eos>']])
    df_sample['pol_ids'] = df_sample.apply(encode_pol_row, axis=1)

df_samples = {key: Data.trim_by_ids(df_sample, ['eng_ids', 'pol_ids'], max_len=35) for key, df_sample in df_samples.items()}

Dropped 0 / 100 rows exceeding max_len=35
Dropped 0 / 100 rows exceeding max_len=35
Dropped 0 / 100 rows exceeding max_len=35
Dropped 0 / 100 rows exceeding max_len=35


## **v1_3.** BLEU Per Epoch **(100 x 4)**

In [147]:
epoch_rows = []
print(f"{'||':>12}   self_F   |   self_M   |   self_NA  | self_OTHER ||  avg_bleu  ||")
for epoch in range(7, 16):
    evaluator = Tests.BleuEvaluation(checkpoint_path=f'../checkpoints/translator_v1_3-{epoch}.pt',
                                      artifacts_path='../local_data/model_reference/translator_v1_3/',
                                      hyperparams=(512, 1024, 8, 4, 0.3, 37), n_ref=1)

    ref_means = {key: mean(evaluator.score_series(df_sample, num_k=4)) for key, df_sample in df_samples.items()}
    row = {'epoch': epoch, 'avg_bleu': mean(ref_means.values()), **ref_means}
    epoch_rows.append(row)
    print(f"Epoch: {row['epoch']:>2} ||  {row['self_F']:.6f}  |  {row['self_M']:.6f}  |  {row['self_NA']:.6f}  |  {row['self_NA_OTHER']:.6f}  ||  {row['avg_bleu']:.6f}  ||")

          ||   self_F   |   self_M   |   self_NA  | self_OTHER ||  avg_bleu  ||


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Epoch: 15 ||  0.487163  |  0.404753  |  0.430331  |  0.504467  ||  0.456679  ||


## **v1_3.** Best epoch according to **avg_bleu**

In [148]:
df_epochs = pd.DataFrame(epoch_rows).sort_values('avg_bleu', ascending=False).reset_index(drop=True)
df_epochs.head(5)

,epoch,avg_bleu,self_F,self_M,self_NA,self_NA_OTHER
0,15,0.456679,0.487163,0.404753,0.430331,0.504467
1,14,0.455929,0.476323,0.418106,0.428978,0.500308
2,13,0.449925,0.462216,0.417774,0.431618,0.488093
3,12,0.442812,0.467547,0.412493,0.408585,0.482622
4,11,0.439652,0.444376,0.406061,0.427189,0.480983


## **v1_4.** translator_v1_4

In [225]:
with open('../local_data/translator_v2/data_final_opensub.pkl', 'rb') as f:
    df_data = pickle.load(f)

df_samples = {}
for val in ['F', 'M', 'NA', 'NA_OTHER']:
    df_samples[f'self_{val}'] = df_data[df_data['self_ref'] == val].sample(100, random_state=42).reset_index(drop=True)
for val in ['F', 'M', 'P', 'NA', 'NA_OTHER']:
    df_samples[f'addr_{val}'] = df_data[df_data['addr_ref'] == val].sample(100, random_state=42).reset_index(drop=True)

for df_sample in df_samples.values():
    df_sample['eng_split'] = df_sample['eng_text'].apply(tokenize_eng)
    df_sample['pol_split'] = df_sample['pol_text'].apply(tokenize_pol)

In [226]:
with open('../local_data/model_reference/translator_v1_4/tokenizer_eng.pkl', 'rb') as f:
    tokenizer_eng = pickle.load(f)
with open('../local_data/model_reference/translator_v1_4/tokenizer_pol.pkl', 'rb') as f:
    tokenizer_pol = pickle.load(f)
with open('../local_data/model_reference/translator_v1_4/encoder_eng.pkl', 'rb') as f:
    encoder_eng = pickle.load(f)
with open('../local_data/model_reference/translator_v1_4/encoder_pol.pkl', 'rb') as f:
    encoder_pol = pickle.load(f)

encode_pol_row = make_ref_encoder(tokenizer_pol, encoder_pol, [('self_ref', {'F': '<self_f>', 'M': '<self_m>', 'NA': '<self_na>', 'NA_OTHER': '<self_na>'}),
                       ('addr_ref', {'F': '<addr_f>', 'M': '<addr_m>', 'P': '<addr_p>', 'NA': '<addr_na>', 'NA_OTHER': '<addr_na>'})])

for df_sample in df_samples.values():
    df_sample['eng_ids'] = df_sample['eng_split'].apply(lambda tab: encoder_eng.encode_snt(tab) + [tokenizer_eng.vocab['<eos>']])
    df_sample['pol_ids'] = df_sample.apply(encode_pol_row, axis=1)

df_samples = {key: Data.trim_by_ids(df_sample, ['eng_ids', 'pol_ids'], max_len=26) for key, df_sample in df_samples.items()}

Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 1 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 2 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26


## **v1_4.** BLEU Per Epoch **(100 x 4)**

In [227]:
epoch_rows = []
print(f"{'||':>12} self_F  | self_M  | self_NA | self_OTH || addr_F  | addr_M  | addr_P  | addr_NA | addr_OTH ||  AVG_BLEU  ||")
for epoch in range(7, 16):
    evaluator = Tests.BleuEvaluation(checkpoint_path=f'../checkpoints/translator_v1_4-{epoch}.pt',
                                      artifacts_path='../local_data/model_reference/translator_v1_4/',
                                      hyperparams=(512, 1024, 8, 4, 0.3, 27), n_ref=2)

    ref_means = {key: mean(evaluator.score_series(df_sample, num_k=4)) for key, df_sample in df_samples.items()}
    row = {'epoch': epoch, 'avg_bleu': mean(ref_means.values()), **ref_means}
    epoch_rows.append(row)
    print(f"Epoch: {row['epoch']:>2} || {row['self_F']:.5f} | {row['self_M']:.5f} | {row['self_NA']:.5f} | {row['self_NA_OTHER']:.5f}  ||"
      f" {row['addr_F']:.5f} | {row['addr_M']:.5f} | {row['addr_P']:.5f} | {row['addr_NA']:.5f} | {row['addr_NA_OTHER']:.5f}  ||  {row['avg_bleu']:.6f}  ||")

          || self_F  | self_M  | self_NA | self_OTH || addr_F  | addr_M  | addr_P  | addr_NA | addr_OTH ||  AVG_BLEU  ||


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Epoch: 15 || 0.33578 | 0.34075 | 0.30275 | 0.30894  || 0.32705 | 0.35582 | 0.31590 | 0.34619 | 0.33146  ||  0.329405  ||


## **v1_4.** Best epoch according to **avg_bleu**

In [228]:
df_epochs = pd.DataFrame(epoch_rows).sort_values('avg_bleu', ascending=False).reset_index(drop=True)
df_epochs.head(5)

,epoch,avg_bleu,self_F,self_M,self_NA,self_NA_OTHER,addr_F,addr_M,addr_P,addr_NA,addr_NA_OTHER
0,13,0.332268,0.349742,0.346291,0.291932,0.335211,0.301774,0.362368,0.322236,0.341493,0.339362
1,15,0.329405,0.335780,0.340749,0.302754,0.308937,0.327055,0.355815,0.315899,0.346191,0.331464
2,11,0.328695,0.330609,0.340165,0.312149,0.313303,0.304991,0.375281,0.317188,0.338176,0.326395
3,14,0.328269,0.328044,0.341243,0.308771,0.330493,0.309812,0.366155,0.327226,0.323113,0.319561
4,12,0.326484,0.338848,0.348843,0.299075,0.309376,0.301267,0.371647,0.319316,0.319619,0.330368


## **7.** translator_v1_5

In [229]:
with open('../local_data/model_reference/translator_v1_5/tokenizer_eng.pkl', 'rb') as f:
    tokenizer_eng = pickle.load(f)
with open('../local_data/model_reference/translator_v1_5/tokenizer_pol.pkl', 'rb') as f:
    tokenizer_pol = pickle.load(f)
with open('../local_data/model_reference/translator_v1_5/encoder_eng.pkl', 'rb') as f:
    encoder_eng = pickle.load(f)
with open('../local_data/model_reference/translator_v1_5/encoder_pol.pkl', 'rb') as f:
    encoder_pol = pickle.load(f)

encode_pol_row = make_ref_encoder(tokenizer_pol, encoder_pol, [('self_ref', {'F': '<self_f>', 'M': '<self_m>', 'NA': '<self_na>', 'NA_OTHER': '<self_na>'}),
                       ('addr_ref', {'F': '<addr_f>', 'M': '<addr_m>', 'P': '<addr_p>', 'NA': '<addr_na>', 'NA_OTHER': '<addr_na>'})])

for df_sample in df_samples.values():
    df_sample['eng_ids'] = df_sample['eng_split'].apply(lambda tab: encoder_eng.encode_snt(tab) + [tokenizer_eng.vocab['<eos>']])
    df_sample['pol_ids'] = df_sample.apply(encode_pol_row, axis=1)

df_samples = {key: Data.trim_by_ids(df_sample, ['eng_ids', 'pol_ids'], max_len=26) for key, df_sample in df_samples.items()}

Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 99 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 98 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26
Dropped 0 / 100 rows exceeding max_len=26


## **8.** BLEU Per Epoch

In [230]:
epoch_rows = []
print(f"{'||':>12} self_F  | self_M  | self_NA | self_OTH || addr_F  | addr_M  | addr_P  | addr_NA | addr_OTH ||  AVG_BLEU  ||")
for epoch in range(7, 16):
    evaluator = Tests.BleuEvaluation(checkpoint_path=f'../checkpoints/translator_v1_5-{epoch}.pt',
                                      artifacts_path='../local_data/model_reference/translator_v1_5/',
                                      hyperparams=(512, 1024, 8, 4, 0.3, 27), n_ref=2)

    ref_means = {key: mean(evaluator.score_series(df_sample, num_k=4)) for key, df_sample in df_samples.items()}
    row = {'epoch': epoch, 'avg_bleu': mean(ref_means.values()), **ref_means}
    epoch_rows.append(row)
    print(f"Epoch: {row['epoch']:>2} || {row['self_F']:.5f} | {row['self_M']:.5f} | {row['self_NA']:.5f} | {row['self_NA_OTHER']:.5f}  ||"
          f" {row['addr_F']:.5f} | {row['addr_M']:.5f} | {row['addr_P']:.5f} | {row['addr_NA']:.5f} | {row['addr_NA_OTHER']:.5f}  ||  {row['avg_bleu']:.6f}  ||")

          || self_F  | self_M  | self_NA | self_OTH || addr_F  | addr_M  | addr_P  | addr_NA | addr_OTH ||  AVG_BLEU  ||


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Epoch: 15 || 0.33274 | 0.34014 | 0.32244 | 0.35028  || 0.28899 | 0.35617 | 0.31885 | 0.35062 | 0.34675  ||  0.334108  ||


In [231]:
df_epochs = pd.DataFrame(epoch_rows).sort_values('avg_bleu', ascending=False).reset_index(drop=True)
df_epochs.head(5)

,epoch,avg_bleu,self_F,self_M,self_NA,self_NA_OTHER,addr_F,addr_M,addr_P,addr_NA,addr_NA_OTHER
0,13,0.338231,0.338437,0.349543,0.321856,0.346049,0.304201,0.361720,0.323502,0.349732,0.349037
1,14,0.337880,0.326600,0.342041,0.331343,0.337814,0.305917,0.380216,0.319784,0.352447,0.344757
2,15,0.334108,0.332736,0.340139,0.322435,0.350278,0.288995,0.356168,0.318852,0.350620,0.346746
3,12,0.332288,0.339171,0.328752,0.311795,0.329248,0.307404,0.366138,0.327643,0.356814,0.323627
4,11,0.326078,0.330278,0.311110,0.317238,0.333302,0.307397,0.356182,0.320054,0.348918,0.310226
